# 03 Redis 持久化

## 八股问答

### 1. Redis 宕机如何处理？

先区分宕机原因：

- 进程崩溃：重启 Redis，根据 RDB/AOF 恢复数据，再检查日志和内存压力。
- 操作系统崩溃或断电：依赖落盘文件恢复，最后落盘之后的写入可能丢失。
- 单点故障：配合哨兵或 Cluster 自动切换，减少业务停机。
- 数据丢失：比较最后一个 RDB 时间和 AOF 的 `fsync` 策略，确定可接受的丢失窗口。

处理顺序：

1. 确认主从、日志和监控状态。
2. 重启或触发故障转移。
3. 检查 RDB/AOF 文件是否完整。
4. 用 `redis-check-rdb`、`redis-check-aof` 检查文件。
5. 恢复后验证数据和服务，必要时从备份补齐。

生产环境通常同时配置 RDB + AOF，并定期做异地备份；备份必须能恢复到冷环境，而不能只依赖运行中的进程。

### 2. AOF 和 RDB 的优劣对比？

| 维度 | RDB | AOF |
| --- | --- | --- |
| 记录内容 | 某一时刻的内存快照 | 写命令追加日志 |
| 文件大小 | 通常更小 | 通常更大，可 rewrite |
| 恢复速度 | 快，直接加载快照 | 慢，需要重放命令 |
| 数据丢失 | 两次快照之间的修改可能丢失 | 取决于 `appendfsync` 策略，丢失窗口更小 |
| 写入开销 | fork + 写快照 | 每个写命令追加日志 |
| 备份迁移 | 适合作为全量备份 | 适合做增量恢复和审计 |

常用配置：

```text
save 900 1
save 300 10
save 60 10000
appendonly yes
appendfsync everysec
```

`appendfsync` 常见值：

- `always`：每个写命令都 fsync，数据最安全但最慢。
- `everysec`：每秒 fsync 一次，是性能和安全的折中，可能丢失 1 秒数据。
- `no`：交给操作系统决定，速度最快但丢失窗口更大。

### 3. 混合持久化是什么？

Redis 4.0 之后支持 **RDB + AOF 混合持久化**，即 AOF rewrite 时先生成 RDB 快照作为前缀，
再追加 rewrite 期间的增量命令。

```text
混合 AOF 文件 = RDB 快照 + 后续增量命令
```

优点：

- 加载比纯 AOF 快，因为前半部分直接加载 RDB。
- 比纯 RDB 丢失窗口小，因为后半部分保留增量命令。
- 适合频繁写、希望兼顾恢复速度和数据安全性的场景。

配置：

```text
appendonly yes
aof-use-rdb-preamble yes
```

混合文件不能再用纯文本方式查看，恢复时由 Redis 自动识别前半段 RDB 和后半段 AOF。

### 4. 执行快照时，数据能修改吗？

可以。RDB 快照通过 `fork` 创建子进程，主线程继续处理读写请求。

底层依赖 **写时复制（Copy-On-Write）**：

- fork 时父子进程共享同一份物理内存页。
- 主线程修改某个页时，先复制旧页，子进程继续使用快照页。
- 子进程把当前内存状态写成 RDB，主线程不会被快照全程阻塞。

风险在于内存大、写入频繁时，fork 需要复制页表，且 COW 会额外占用内存；内存接近上限时
可能触发 OOM。日常建议控制内存水位、降低快照频率，或使用 `BGSAVE` 并监控 fork 耗时。

### 5. Redis 中 key 过大对持久化的影响？

大 Key 会带来以下问题：

- RDB 快照需要复制和写入更多数据，落盘时间变长。
- AOF rewrite 和命令追加也会处理大值，文件膨胀更快。
- `fork` 之后写大对象，COW 可能复制更多旧页，瞬时内存开销变大。
- 大 Key 可能阻塞单线程命令，造成查询延迟和超时。
- 备份和迁移时占用更多网络和磁盘 IO。

建议：

- 拆分大 Key，例如把一个大 Hash 拆成多个小 Hash。
- 控制 String 大小，避免把大文件或长文本直接塞进 Redis。
- 使用 `SCAN` 而不是 `KEYS` 检查，避免阻塞。
- 监控 `MEMORY USAGE`、`BIGKEYS` 和慢命令。
- 大 Key 即使有 TTL，过期时也可能造成瞬时删除开销，需要分批删除。


## 面试话术

“我会同时开启 RDB 和 AOF。RDB 适合快速恢复和备份，AOF 提供更小的丢失窗口；
生产环境常用 `appendfsync everysec`，并开启混合持久化，让 AOF rewrite 包含 RDB 前缀，
兼顾恢复速度和安全性。快照时主线程仍可写，底层靠 fork 和写时复制；如果内存接近上限，
大 Key 和频繁写会放大 COW 和落盘压力，所以我会控制大 Key、拆分数据并监控内存水位。”


## 自查清单

- [ ] 能比较 RDB 和 AOF 的丢失窗口、文件大小和恢复速度
- [ ] 能说明混合持久化为什么兼顾两者
- [ ] 能解释 fork 后为什么还能写入
- [ ] 能说明大 Key 对持久化的影响及处理方式
